# SOLUCIÓN PROBLEMA 2 - FINAL Probabilidad y Estadística - iMAT-ICAI - 19 de mayo 2023

# Vale el 40% de la nota del examen
---

NO CAMBIES EL NOMBRE AL NOTEBOOK, SIMPLEMENTE ESCRIBE AQUÍ TUS DATOS
### Nombre: RELLENAR
### Apellidos: RELLENAR
---

### En un estudio sobre el clima de unas islas poco conocidas se puede observar que julio, agosto y septiembre son los tres meses con menor probabilidad de lluvias, mientras que en el resto de 9 meses es más fácil que llueva. Se considera que todos los meses del año son de 30 días y que los días son unos independientes de otros.

### Las precipitaciones son frecuentes todo el año. La probabilidad de que llueva un día en los meses de julio, agosto o septiembre (grupo de meses 3M) es del 40%, mientras que el resto de meses (grupo de meses 9M) es del 80%. Un día lluvioso es aquel en el que llueve algo en algún momento del día, mientras que un día no lluvioso es en el que no llueve nada.

### Además, según dicho estudio, sabiendo que ha llovido un día, entonces el nº de horas que llueve ese día se puede modelar bastante bien con una v.a. diferente dependiendo del mes. Si X es la v.a. que representa el "nº de horas que llueve en un día lluvioso", entonces:

* ### Grupo de meses 3M: la v.a. X se distribuye como una binomial de parámetros p = 0.4 y n = 24
* ### Grupo de meses 9M: la v.a. X se distribuye como una binomial de parámetros p = 0.7 y n = 24

### Nota: Según el modelo propuesto puede ocurrir que el nº de horas que llueve en un día lluvioso sea 0, que podría parecer ilógico, pero es un suceso muy raro con los parámetros utilizados en las distribuciones binomiales.

Responde las preguntas utilizando instrucciones print() para mostrar claramente tu respuesta en la ejecución de la celda, explicando cómo lo resuelves. Utiliza comentarios en tu código para facilitar su lectura. No escribas líneas de código muy largas para facilitar su lectura.

### Importamos todos los paquetes aquí por comodidad

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy import optimize as opt  # para estimación por máxima verosimilitud

from statsmodels.distributions.empirical_distribution import ECDF

import statsmodels.api as sm
from statsmodels.nonparametric.kde import KDEUnivariate

from anova import anova1, anova2

### a) Calcula la probabilidad de que un día cualquiera sea NO lluvioso. Justifica la respuesta (1 punto)

In [ ]:
# Respuesta

print('Un día puede ser lluvioso (LL) o no lluvioso (NLL). Se pide calcular P(NLL).')

p_NLL_cond_3M = 0.6
p_NLL_cond_9M = 0.2
print(f'La P(NLL|3M) = 1 - P(LL|3M) = {p_NLL_cond_3M}, mientras que P(NLL|9M) = 1 - P(LL|9M) = {p_NLL_cond_9M}')
print('Todos los meses son de 30 días, por lo que se puede considerar que hay una partición con dos grupos: 3M y 9M:')
print('P(3M) = 3/12 y P(9M)=9/12.')

print('La probabilidad pedida se calcula a partir del teorema de la probabilidad total:')
print('P(NLL) = P(NLL|3M) * P(3M) +  P(NLL|9M) * P(9M)')
p_NLL = p_NLL_cond_3M * 3/12 + p_NLL_cond_9M * 9/12
print(f'P(NLL) = {p_NLL_cond_3M} * 3/12 + {p_NLL_cond_9M} * 9/12 = {p_NLL}')

print(f'Por tanto, la probabilidad de que un día cualquiera sea NO lluvioso vale {p_NLL}.')

### b) Calcula la probabilidad de que llueva al menos 25 días en dos meses cualesquiera del tipo 3M. Justifica la respuesta (0.5 puntos)

In [ ]:
# Respuesta

p_LL_cond_3M = 0.4

print('Se pide calcular P(NdiasLL2m3M>=25), siendo NdiasLL2m9M la v.a. que representa el nº de días que llueve en dos meses del tipo 3M.')
print('Dado que los días son independientes y que un día puede ser lluvioso o no, NdiasLL2m3M ~ Binomial de parámetros:')
print(f'  - Probabilidad de éxito (llueve siendo un día de 3M): {p_LL_cond_3M}')
print('  - Nº de repeticiones (2 meses): 60 días')

# crea las v.a. NdiasLL2m9M que siguen una binomial
n= 60
NdiasLL2m3M = stats.binom(n=n,p=p_LL_cond_3M)

prob = 1 - NdiasLL2m3M.cdf(24)
print(f'La probabilidad de que llueva al menos 25 días en dos meses cualesquiera del tipo 3M es P(NdiasLL2m3M>=25) = 1 - cdf(24) = {prob}')

# visualmente para comprobar
print('Gráficamente:')
xs = range(0, 60)
xs_mayor_igual_25 = range(25, 60)
fig = plt.figure(1, figsize=(12, 4))
plt.stem(xs, NdiasLL2m3M.pmf(xs), basefmt='none')
plt.stem(xs_mayor_igual_25, NdiasLL2m3M.pmf(xs_mayor_igual_25), markerfmt ='C1s', basefmt='none', label='Al menos 25 días')
plt.grid()
plt.legend()
plt.title(f'X3M ~ Binomial(n={n}, p={p_LL_cond_3M})')
plt.show()

### c) Representa en la misma gráfica (en dos subplots diferentes) las funciones de masa de probabilidad de las variables aleatorias que representan el "nº de horas que llueve en un día lluvioso" para los dos tipos de meses (grupos 3M y 9M). Indica el nº de horas más probable en 9M (0.5 puntos)

In [ ]:
# Respuesta

# crea las v.a. "X" que siguen una binomial
n= 24
p3M=0.4
X3M = stats.binom(n=n,p=p3M)
n= 24
p9M=0.7
X9M = stats.binom(n=n,p=p9M)

# puntos en los que se evalua las funciones para representarlas
xs = range(0, 25)

# pinta sincronizadametne ambas funciones
fig = plt.figure(1, figsize=(12, 8))
ax1 = plt.subplot(2,1,1)
plt.stem(xs, X3M.pmf(xs), basefmt='none')
plt.grid()
plt.title(f'X3M ~ Binomial(n={n}, p={p3M})')

plt.subplot(2,1,2, sharex = ax1)
plt.stem(xs, X9M.pmf(xs), basefmt='none')
plt.ylabel('pmf(x)')
plt.grid()
plt.title(f'X9M ~ Binomial(n={n}, p={p9M})')
plt.show()

print(f'El nº de horas más probable en 9M viene dado por la moda de X9M: {17}, es decir, '
     'el nº de horas donde la pmf(X) se hace máxima.')

### d) Calcula la probabilidad de que un día sea del tipo 3M, sabiendo que ha llovido y más de 10 horas. Justifica la respuesta (1 punto)

In [ ]:
# Respuesta

print('Se sabe que el día es lluvioso y que ha llovido más de 10 horas. Se pide la probabilidad a posteriori P(3M|X>10).')
print('Se puede aplicar el teorema de Bayes para calcularla, considerando que 3M y 9M forman una partición.')

prob_X_mayor10_cond_3M = 1 - X3M.cdf(10)
prob_X_mayor10_cond_9M = 1 - X9M.cdf(10)

prob_3M = 3/12
prob_9M = 9/12

prob_3M_y_Xmayor10 = prob_X_mayor10_cond_3M * prob_3M
prob_9M_y_Xmayor10 = prob_X_mayor10_cond_9M * prob_9M

prob_Xmayor10 = prob_3M_y_Xmayor10 + prob_9M_y_Xmayor10

prob_3M_cond_Xmayor10 = prob_3M_y_Xmayor10 / prob_Xmayor10
#prob_9M_cond_Xmayor10 = prob_9M_y_Xmayor10 / prob_Xmayor10

print(f'Por tanto, la probabilidad de que un día sea del tipo 3M, sabiendo que ha llovido y más de 10 horas vale {prob_3M_cond_Xmayor10}') 

### Además, en el estudio también se indica que en un día la precipitación diaria (medida en litros por metro cuadrado) se puede modelar con una v.a. diferente dependiendo del tipo de mes. Si R es la v.a. que representa la "precipitación total en un día lluvioso", entonces:

* ### Grupo de meses 3M: la v.a. R se distribuye como una normal de media 15.5 l/m2 y desviación típica 3.2 l/m2
* ### Grupo de meses 9M: la v.a. R se distribuye como una normal de media 5.3 l/m2 y desviación típica 1.25 l/m2

### e) Calcula la probabilidad de que en un año cualquiera la precipitación acumulada en los tres meses de tipo 3M supere la precipitación acumulada durante el resto de meses del año. Justifica la respuesta (2 puntos)

In [ ]:
# Respuesta

print('Se pide P(R3M > R9M), siendo R3M la precipitación acumulada en 3M y R9M la precipitación acumulada en el resto de 9 meses.')

# distribuciones acumuladas en 3M y 9M
medR3M = 3 * 30 * 15.5
varR3M = 3 * 30 * (3.2)**2
stdR3M = np.sqrt(varR3M)

medR9M = 9 * 30 * 5.3
varR9M = 9 * 30 * (1.25)**2
stdR9M = np.sqrt(varR9M)

print('R3M se distribuye como una normal ya que es la suma 3x30 = 90 v.a. que se distribuyen i.i.d. según una normal. Sus parámetros son:')
print(f'  - media de R3M: 90 * 25.5 = {medR3M} l/m2')
print(f'  - desv. típica de R3M: sqrt(90) * 3.2 = {stdR3M} l/m2')

print('R9M también se distribuye como una normal ya que es la suma 3x30 = 90 v.a. que se distribuyen i.i.d. según una normal. Sus parámetros son:')
print(f'  - media de R9M: 90 * 12.3 = {medR9M} l/m2')
print(f'  - desv. típica de R9M: sqrt(90) * 1.25 = {stdR9M} l/m2')

# definimos v.a. diferencia
mu_difR3M9M = medR3M - medR9M
var_difR3M9M = varR3M + varR9M # se suman
std_difR3M9M = np.sqrt(var_difR3M9M)
difR3M9M = stats.norm(loc = mu_difR3M9M, scale = std_difR3M9M)

print('Para calcular P(R3M > R9M) = P(R3M - R9M > 0), hay que trabajar con la v.a. difR3M9M = R3M - R9M.')
print('La v.a. difR3M9M es también una normal al ser una resta de normales independientes, con parámetros:')
print(f'  - media: {medR3M} - {medR9M} = {mu_difR3M9M} l/m2')
print(f'  - desv. típica: sqrt({varR3M} + {varR9M}) = {std_difR3M9M} l/m2')

prob = 1-difR3M9M.cdf(0)
print('Por tanto, la probabilidad de que la precipitación acumulada en 3M supere a la de 9M es:')
print(f'P(R3M > R9M) = P(R3M - R9M > 0) = 1 - cdf(0) = {prob}')


# comprobación gráfica
R3M = stats.norm(loc = medR3M, scale = stdR3M)
R9M = stats.norm(loc = medR9M, scale = stdR9M)

xs = np.linspace(difR3M9M.ppf(0.001), R9M.ppf(0.999), 1000)
xsMayor0 = np.linspace(0, R9M.ppf(0.999), 1000)

# pinta sincronizadamente ambas funciones
fig = plt.figure(1, figsize=(16, 4))
plt.subplot(2,1,1)
plt.plot( xs, R3M.pdf(xs), color="green", label = 'R3M')
plt.plot( xs, R9M.pdf(xs), color="red", label = 'R9M')
plt.legend()
plt.subplot(2,1,2)
plt.fill_between( xs, difR3M9M.pdf(xs), color="orange", alpha=0.5)
plt.fill_between( xsMayor0, difR3M9M.pdf(xsMayor0), color="green", alpha=0.5, label='Probabilidad pedida')
plt.plot( xs, difR3M9M.pdf(xs), color="orange", label = 'difR3M9M')
plt.legend()
plt.show()


### f) Sea Y la v.a. que representa el nº de horas de lluvia para un día cualquiera del año. Calcula el valor esperado de la v.a. Y. Justifica la respuesta (2 puntos)

In [ ]:
# Respuesta

print('Se pide calcular E[Y]. El valor esperado del nº de horas de lluvia depende del tipo de mes (3M o 9M), así como de si el día llueve o no. ' 
      'Si el día no llueve, entonces el nº de horas de lluvia será nulo, con independencia del tipo de mes. Se tienen los siguientes casos:')

print('  - caso 3M (ocurre 3/12)')
print('       - Si llueve (ocurre con probabilidad 0.4) entonces E[Y] = E[X3M]')
print('       - Si no llueve (ocurre con probabilidad 0.6) entonces E[Y] = 0')

print('  - caso 9M (ocurre 9/12)')
print('       - Si llueve (ocurre con probabilidad 0.8) entonces E[Y] = E[X9M]')
print('       - Si no llueve (ocurre con probabilidad 0.2) entonces E[Y] = 0')

# se puede calcular también como n * p para cada variable
E_X3M = X3M.mean() 
E_X9M = X9M.mean() 

print(f'Cuando llueve, el valor esperado del nº de horas de lluvia en un día de 3M es E[3M] = {E_X3M}, mientras que el valor esperado del nº de horas en el '
      f'resto de meses (9M) es E[9M] = {E_X9M}')

print('Utilizando la ley de la esperanza total se puede ver que:')

E_Y = 3/12 * 0.4 *  E_X3M + 9/12 * 0.8 * E_X9M
print(f'E[Y] = 3/12 * 0.4 *  E[X3M] + 0 + 9/12 * 0.8 *  E[X9M] + 0 = {3/12 * 0.4 *  E_X3M} + {9/12 * 0.8 * E_X9M} = {E_Y} horas diarias.')


### g) Ahora, para confirmar el resultado anterior, vuelve a calcular el valor esperado de la v.a. Y "nº de horas de lluvia para un día cualquiera del año" de forma empírica mediante simulación. Para ello genera 100 años de 12 meses x 30 días (36000 nº aleatorios) y utiliza esos datos simulados para estimar el valor esperado de la v.a. Y. Tienes que generar un vector "y" con 36000 valores aleatorios correspondientes al nº de horas de lluvia para cada día de esos 10 años a partir de las distribuciones implicadas y probabilidades dadas. Básate en la plantilla de código que se muestra a continuación. Justifica la respuesta (1 punto)

In [ ]:
# Ejemplo ilustrativo del tipo de código a utilizar

# fijamos la semilla
semilla = 9999
np.random.seed(semilla)

# variables ficticias
VA1 = stats.binom(n=199,p=0.5)
VA2 = stats.binom(n=9,p=0.1)
VA3 = stats.binom(n=49,p=0.3)

# calcula nº de datos a simular de cada tipo (nº enteros)
n_datos_sim_t1 = int(np.round(0.3 * 300))
n_datos_sim_t2 = int(np.round(0.4 * 300))
n_datos_sim_t3 = int(np.round(0.2 * 300))
n_datos_sim_t4 = int(np.round(0.1 * 300))

# genera rvs de cada tipo
datos_t1 = VA1.rvs(size = n_datos_sim_t1)
datos_t2 = VA2.rvs(size = n_datos_sim_t2)
datos_t3 = VA3.rvs(size = n_datos_sim_t3)
datos_t4 = np.zeros(n_datos_sim_t4) # <--- así se consiguen "n_datos_sim_t4" valores con valor cero

# concatena los datos en un único vector para trabajar con él (la posición en el vector es irrelevante)
datos = np.hstack([datos_t1, datos_t2, datos_t3, datos_t4])

# pinta para confirmar datos tienen buena pinta
fig = plt.figure(1, figsize=(16, 4))
plt.plot(datos)
plt.grid()
plt.ylabel('valor simulado')
plt.xlabel('dato')
plt.title('Datos ficticios inventados')
plt.show()

In [ ]:
# Respuesta

# fijamos la semilla
semilla = 9999
np.random.seed(semilla)

print('Para cada año hay que distinguir entre los meses para generar un nº de horas de lluvia acordes a los datos proporcionados, teniendo '
      'en cuenta también que puede no llover.')

print('Lo más sencillo, ya que el orden de los días no importa, es calcular cuántos días en los 100 años hay que crear de cada tipo (3M, 9M) y '
      'dentro de cada tipo el nº de días que llueve y los que no llueve.')

# número de días del tipo 3M a simular en 10 años
n_anios = 100
n_dias_sim_3M_llueve = int(np.round(3 * 30 * 0.4 * n_anios))
n_dias_sim_3M_no_llueve = int(np.round(3 * 30 * (1-0.4) * n_anios))

# número de días del tipo 3M a simular en 10 años
n_dias_sim_9M_llueve = int(np.round(9 * 30 * 0.8 * n_anios))
n_dias_sim_9M_no_llueve = int(np.round(9 * 30 * (1-0.8) * n_anios))

print('Nº de días a simular de cada tipo en 100 años:')
print('  - Caso 3M (ocurre 3/12)')
print(f'       - Nº de días que llueve: {n_dias_sim_3M_llueve}')
print(f'       - Nº de días que no llueve: {n_dias_sim_3M_no_llueve}')
print('  - Caso 9M (ocurre 9/12)')
print(f'       - Nº de días que llueve: {n_dias_sim_9M_llueve}')
print(f'       - Nº de días que no llueve: {n_dias_sim_9M_no_llueve}')

# genera números aleatorios segun cada una de las dos binomiales, primero días lluvia y lluego los que no
x3m_LL  = X3M.rvs(size = n_dias_sim_3M_llueve)
x3m_NLL = np.zeros(n_dias_sim_3M_no_llueve)

x9m_LL  = X9M.rvs(size = n_dias_sim_9M_llueve)
x9m_NLL = np.zeros(n_dias_sim_9M_no_llueve)

# concatena todos los datos en un único vector para hacer la media (el orden no es importante)
y = np.hstack([x3m_LL, x3m_NLL, x9m_LL, x9m_NLL])

media_y = np.mean(y)

# pinta para confirmar datos tienen buena pinta
fig = plt.figure(1, figsize=(16, 6))
plt.plot(y, label = 'Todos los días')
plt.plot(np.hstack([x3m_LL, x3m_NLL]), '.', color = 'red', label = 'días 3M')
plt.grid()
plt.ylabel('Nº horas simuladas de lluvia')
plt.xlabel('día (100 años)')
plt.title(f'La media de los datos simulados es {media_y}')
plt.legend()
plt.show()

print('A partir de los datos simulados, el valor estimado para el valor esperado de la v.a. "nº de horas de lluvia para un día cualquiera del año" '
      f'vale {media_y}, similar al valor calculado analíticamente.')

# comprobación adicional primer apartado

prob_dia_no_lluvioso = (n_dias_sim_3M_no_llueve +  n_dias_sim_9M_no_llueve) / (n_dias_sim_3M_no_llueve +  n_dias_sim_9M_no_llueve + n_dias_sim_3M_llueve + n_dias_sim_9M_llueve)
print('\nSe puede aprovechar para chequear que la probabilidad de que un día cualquiera no llueva, calculada '
      f'anteriormente, coincide con el cálculo sencillo a partir del nº de días simulados (frecuencias): {prob_dia_no_lluvioso}')

# pinta para confirmar datos tienen buena pinta
print('\nEs posible representar fácilmente también la distribución que se observa a partir de los datos simulados '
      '(calculando frecuencias mediante un histograma con muchos bins):')
fig = plt.figure(1, figsize=(12, 3))
plt.hist(y, bins = 200, label = 'Histograma datos simulados')
plt.plot(media_y, 0,  'o', markersize = 10, color='red', label = f'Media = {media_y}')
plt.grid()
plt.legend()
plt.xlabel('nº de horas de lluvia para un día cualquiera del año')
plt.show()

### En el estudio también incluye un enlace a github donde se pueden descargar los datos de las precipitaciones diarias (variable RAIN) recogidas en la estacióm meteorológica situada en la isla durante los últimos 10 años. Los datos están adaptados para ser meses de 30 días.

In [ ]:
# Carga los datos descargados de github

# cargamos los datos de la muestra disponible
df = pd.read_csv('muestraPrecipitacion.csv')
print(df)

# pinta para confirmar datos tienen buena pinta
fig = plt.figure(1, figsize=(16, 8))
plt.subplot(2,1,1)
plt.plot(df['RAIN'])
plt.grid()
plt.ylabel('Precipitación (l/m2)')
plt.title(f'Evolución temporal de RAIN')
plt.subplot(2,1,2)
plt.plot(df['NUM_HOURS_RAIN'])
plt.grid()
plt.ylabel('Nº de horas lluvia')
plt.xlabel('día (10 años)')
plt.title(f'Evolución temporal de NUM_HOURS_RAIN')
plt.show()

### h) Analiza si el tipo de distribución indicada de la v.a. "precipitación total en un día lluvioso en un mes del tipo 3M", una normal de media 15.5 l/m2 y desviación típica 3.2 l/m2, es coherente con los datos empíricos disponibles en github. Justifica la respuesta utilizando un contraste de hipótesis no paramétrico, razonando con un nivel de significación del 1%. (1 punto)

In [ ]:
# Datos correspondientes a precipitación en un día lluvioso en un mes del tipo 3M

r3m = df['RAIN'].loc[(df['TYPE_MONTH'] == '3M') & (df['RAIN'] > 0)]  #  <--------   trabaja con esta variable

# pinta para confirmar datos tienen buena pinta
fig = plt.figure(1, figsize=(12, 3))
plt.hist(r3m, bins = 40)
plt.xlabel('Precipitación r3m (l/m2)')
plt.show()

In [ ]:
# Respuesta

mean_r3m = np.mean(r3m)
std_r3m = np.std(r3m)

print(f'Los valores de la media {mean_r3m} y desviación típica {std_r3m} de la muestra son similares a los parámetros indicados de la normal.')

print('Para verificar si los datos de la muestra siguen realmente una normal, lo pedido, se realiza un contraste no paramétrico KS.')

distribucion_teo = stats.norm

# nivel de significación
alfa = 0.01

# realiza el test
ks = sm.stats.diagnostic.kstest_normal(r3m, dist='norm', pvalmethod='table')

max_d   = ks[0]
p_valor = ks[1]
print(f'   Máxima distancia observada = {max_d} -> p_valor = {p_valor}')

# concluye
print(f'Como el pvalor >= {alfa}, no se puede rechazar H0 con un nivel de significación del {alfa*100}% ya que la distancia máxima observada no es suficientemente grande.')
print(f'Según el test no paramétrico KS se puede afirmar, con un nivel de significación del {alfa*100}%, que los datos de la muestra siguen una distribución normal.')

print('Además confirmamos gráficamente que los datos reales siguen una distribución normal con un qqplot:')

# 1.3. Gráfico qqplot con contraste KS
titulo = f'KS test:  p_valor = {p_valor}'

# genera qqplot de la variable x añadiendo en el título el resultado del KS
sm.qqplot(r3m, line='45', fit=True, dist=distribucion_teo)
plt.title(titulo)
plt.grid(); plt.show()

print('Salvo en la cola de la derecha que podría haber cierta duda, el grueso de la muestra están alineados con la recta, como era de esperar visto el resultado de KS.')

### i) Realiza un ANOVA para determinar si la variable RAIN cambia significativamente con el mes (MONTH), el año (YEAR) o su interacción. Concluye claramente sobre el resultado obtenido con un nivel de significación del 1%. Muestra gráficamente los coeficientes del modelo de medias y comenta lo que observas. Justifica la respuesta (1 punto)

In [ ]:
# Respuesta

# modelo con interacción
salida = 'RAIN'
factor_1 = 'MONTH'
factor_2 = 'YEAR'
formula = f'{salida} ~ C({factor_1}) * C({factor_2})'
print('Fórmula: ', formula)

# calcula ANOVA 2 (CON interacción)
tabla_anova, modelo_medias = anova2(formula, df) 

# muestra tabla anova
print('Tabla ANOVA:')
print(tabla_anova)
print('\n')

# concluye
print('Observando los p-valores es evidente que con los niveles de significación habituales se puede afirmar que la variable '
      'RAIN solo depende de MONTH (pvalor muy pequeño), los pvalores para YEAR y la interacción MONTH*YEAR son mayores del 20%, '
      'no se puede rechazar la hipótesis nula asociada.')

In [ ]:
# Representación gráfica del modelo de medias

modelo_medias_not_inter = (
    modelo_medias
    .loc[lambda a: ~a.index.str.contains(":")]
)

modelo_medias_inter = (
    modelo_medias
    .loc[lambda a: a.index.str.contains(":")]
    .assign(
        nombre_f1 = lambda a: a.index.str.split(" : ").str[0],
        nombre_f2 = lambda a: a.index.str.split(" : ").str[1]
    )
)

ind_factor_1 = np.where(modelo_medias_not_inter.index.str.startswith(factor_1))[0].tolist()
ind_factor_2 = np.where(modelo_medias_not_inter.index.str.startswith(factor_2))[0].tolist()

gr_factor_1 = modelo_medias_not_inter.index[ind_factor_1].str.replace('.*_', '', regex =True)
gr_factor_2 = modelo_medias_not_inter.index[ind_factor_2].str.replace('.*_', '', regex =True)


fig = plt.figure(1, figsize=(20, 3))
plt.subplot(1,3,1)
plt.bar(modelo_medias_not_inter.index[0], modelo_medias_not_inter.Mean.iloc[0])
plt.ylabel('Media global')

plt.subplot(1,3,2)
plt.bar(gr_factor_1, modelo_medias_not_inter.Mean.iloc[ind_factor_1])
plt.ylabel('Coeficientes alfa')
plt.xlabel(factor_1)

plt.subplot(1,3,3)
plt.bar(gr_factor_2, modelo_medias_not_inter.Mean.iloc[ind_factor_2])
plt.ylabel('Coeficientes beta')
plt.xlabel(factor_2)
plt.show()

fig = plt.figure(2, figsize=(80, 3))
plt.subplot(1,3,1)
plt.bar(modelo_medias_inter.index, modelo_medias_inter.Mean)
plt.ylabel('Coeficientes theta')
plt.xticks(rotation=90)
plt.show()

# comentario
print('Se puede observar que los coeficientes alfa son positivos para los meses 7, 8 y 9 (los del tipo 3M). Esto tiene sentido ya que '
      'según la muestra de github es evidente que el valor esperado de lluvia en un día de 3M es mucho mayor que en un día de 9M.')
print('Las variaciones con el año y la interacción son mucho más pequeñas, también lógico considerando el resultado del contraste.')